In [1]:
import duckdb
import pandas as pd
from pathlib import Path
from datetime import datetime, timezone, timedelta
import pytz
from google.transit import gtfs_realtime_pb2
from google.protobuf.message import DecodeError

PROJECT_ROOT = Path(r'C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework')
DB_PATH      = PROJECT_ROOT / 'data' / 'transit_kpi.duckdb'
ARCHIVE_DIR  = PROJECT_ROOT / 'data' / 'raw' / 'gtfs_rt'
STATIC_DIR   = PROJECT_ROOT / 'data' / 'google_bus'

EASTERN = pytz.timezone('America/New_York')

con = duckdb.connect(str(DB_PATH))

print(f'Project root:  {PROJECT_ROOT}')
print(f'Database:      {DB_PATH}')
print(f'Archive dir:   {ARCHIVE_DIR}')
print()
print('Existing tables:')
print(con.sql('SHOW TABLES').df().to_string(index=False))

Project root:  C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework
Database:      C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework\data\transit_kpi.duckdb
Archive dir:   C:\Users\Tobi\Desktop\CSHW\TransitKPIFramework\data\raw\gtfs_rt

Existing tables:
              name
          calendar
    calendar_dates
            gtfsrt
otp_specifications
        stop_times
       trip_delays
             trips


In [2]:
def snapshot_to_service_date(snapshot_ts_utc: datetime) -> tuple[str, int]:
    """
    Convert a UTC snapshot timestamp to (service_date, midnight_unix).

    Uses the GTFS service-day convention: the service day rolls over at 4 AM
    Eastern, not at midnight. A snapshot taken at 12:30 AM Eastern is assigned
    to the previous calendar day's service date --- this is necessary because
    late-night day-service trips can bleed past midnight, and their stops
    are scheduled in static GTFS as 24:xx times on the previous service date.

    Returns:
      service_date_str  --- ISO date string, e.g. '2026-04-07'
      midnight_unix     --- Unix timestamp of Eastern midnight on that
                            service date (handles DST automatically)
    """
    snapshot_eastern = snapshot_ts_utc.astimezone(EASTERN)

    # Rolling 4 AM service-day boundary
    if snapshot_eastern.hour < 4:
        service_date = snapshot_eastern.date() - timedelta(days=1)
    else:
        service_date = snapshot_eastern.date()

    service_date_str = service_date.isoformat()

    # Eastern midnight of the assigned service date --- DST-aware via pytz.localize
    midnight = EASTERN.localize(
        datetime(service_date.year, service_date.month, service_date.day, 0, 0, 0)
    )
    midnight_unix = int(midnight.timestamp())

    return service_date_str, midnight_unix


# Sanity checks
# April 7 2026, 2:30 PM UTC == 10:30 AM EDT --- service date is April 7
test_daytime = datetime(2026, 4, 7, 14, 30, 0, tzinfo=timezone.utc)
assert snapshot_to_service_date(test_daytime) == ('2026-04-07', 1775534400), \
    'Daytime conversion failed'

# April 8 2026, 5:30 AM UTC == 1:30 AM EDT --- still April 7's service day
test_post_midnight = datetime(2026, 4, 8, 5, 30, 0, tzinfo=timezone.utc)
assert snapshot_to_service_date(test_post_midnight) == ('2026-04-07', 1775534400), \
    'Post-midnight rollover failed'

# April 8 2026, 8:30 AM UTC == 4:30 AM EDT --- belongs to April 8
test_after_4am = datetime(2026, 4, 8, 8, 30, 0, tzinfo=timezone.utc)
assert snapshot_to_service_date(test_after_4am) == ('2026-04-08', 1775620800), \
    '4 AM threshold failed'

print('All service-date conversions correct:')
print(f'  10:30 AM EDT April 7  -> {snapshot_to_service_date(test_daytime)}')
print(f'  1:30 AM EDT April 8   -> {snapshot_to_service_date(test_post_midnight)}')
print(f'  4:30 AM EDT April 8   -> {snapshot_to_service_date(test_after_4am)}')

All service-date conversions correct:
  10:30 AM EDT April 7  -> ('2026-04-07', 1775534400)
  1:30 AM EDT April 8   -> ('2026-04-07', 1775534400)
  4:30 AM EDT April 8   -> ('2026-04-08', 1775620800)


In [3]:
# Create gtfsrt table if it doesn't exist with explicit schema
# Using INSERT INTO instead of CREATE TABLE AS lets us append days incrementally
con.execute("""
    CREATE TABLE IF NOT EXISTS gtfsrt (
        snapshot_ts            VARCHAR,
        service_date           VARCHAR,
        midnight_unix          BIGINT,
        trip_id                VARCHAR,
        route_id               VARCHAR,
        schedule_relationship  INTEGER,
        stop_sequence          INTEGER,
        stop_id                VARCHAR,
        predicted_unix         BIGINT,
        predicted_ts           VARCHAR
    )
""")

# Find which service dates are already in the table --- skip files for those dates
existing_dates = set(
    con.sql('SELECT DISTINCT service_date FROM gtfsrt').df()['service_date'].tolist()
)
print(f'Service dates already in gtfsrt: {len(existing_dates)}')
if existing_dates:
    print(f'  range: {min(existing_dates)} to {max(existing_dates)}')

Service dates already in gtfsrt: 9
  range: 2026-04-04 to 2026-04-12


In [4]:
pb_files = sorted(ARCHIVE_DIR.glob('*.pb'))
print(f'Total .pb files in archive: {len(pb_files)}')

# Group files by service_date using the rolling 4 AM boundary
# All daytime AND post-midnight snapshots are kept --- post-midnight ones
# get assigned to the previous service date so late-night trips that bleed
# past midnight are still matchable to their (previous-day) calendar entry
files_by_date: dict[str, list[Path]] = {}
skipped_unparseable_name = 0

for pb_file in pb_files:
    try:
        snapshot_ts_utc = datetime.strptime(
            pb_file.stem, '%Y%m%dT%H%M%SZ'
        ).replace(tzinfo=timezone.utc)
    except ValueError:
        skipped_unparseable_name += 1
        continue

    service_date_str, _ = snapshot_to_service_date(snapshot_ts_utc)
    files_by_date.setdefault(service_date_str, []).append(pb_file)

print(f'Skipped (unparseable filename): {skipped_unparseable_name}')
print(f'Distinct service dates:         {len(files_by_date)}')
if files_by_date:
    print(f'  range: {min(files_by_date)} to {max(files_by_date)}')
    print()
    print('Files per service date:')
    for d in sorted(files_by_date):
        print(f'  {d}: {len(files_by_date[d]):4d} files')

Total .pb files in archive: 29965
Skipped (unparseable filename): 0
Distinct service dates:         22
  range: 2026-04-04 to 2026-04-25

Files per service date:
  2026-04-04:  521 files
  2026-04-05: 1438 files
  2026-04-06: 1438 files
  2026-04-07: 1438 files
  2026-04-08: 1438 files
  2026-04-09: 1439 files
  2026-04-10: 1438 files
  2026-04-11: 1438 files
  2026-04-12: 1438 files
  2026-04-13: 1438 files
  2026-04-14: 1438 files
  2026-04-15: 1439 files
  2026-04-16: 1438 files
  2026-04-17: 1438 files
  2026-04-18: 1439 files
  2026-04-19: 1438 files
  2026-04-20: 1440 files
  2026-04-21: 1438 files
  2026-04-22: 1438 files
  2026-04-23: 1438 files
  2026-04-24: 1439 files
  2026-04-25:  678 files


In [5]:
dates_to_process = sorted(d for d in files_by_date if d not in existing_dates)
print(f'Dates to process: {len(dates_to_process)}')
print(f'Dates skipped (already in table): {len(files_by_date) - len(dates_to_process)}')
print()

failed_files: list[tuple[str, str]] = []
total_records_inserted = 0

for service_date_str in dates_to_process:
    day_files = files_by_date[service_date_str]
    records = []

    for pb_file in day_files:
        try:
            snapshot_ts_utc = datetime.strptime(
                pb_file.stem, '%Y%m%dT%H%M%SZ'
            ).replace(tzinfo=timezone.utc)
            _, midnight_unix = snapshot_to_service_date(snapshot_ts_utc)

            feed = gtfs_realtime_pb2.FeedMessage()
            with open(pb_file, 'rb') as f:
                feed.ParseFromString(f.read())

        except (DecodeError, OSError, ValueError) as e:
            failed_files.append((pb_file.name, f'{type(e).__name__}: {e}'))
            continue

        for entity in feed.entity:
            if not entity.HasField('trip_update'):
                continue

            tu = entity.trip_update
            trip_id   = tu.trip.trip_id
            route_id  = tu.trip.route_id
            sched_rel = tu.trip.schedule_relationship

            for stu in tu.stop_time_update:
                if stu.HasField('arrival'):
                    predicted_time = stu.arrival.time
                elif stu.HasField('departure'):
                    predicted_time = stu.departure.time
                else:
                    continue

                records.append({
                    'snapshot_ts':           snapshot_ts_utc.isoformat(),
                    'service_date':          service_date_str,
                    'midnight_unix':         midnight_unix,
                    'trip_id':               trip_id,
                    'route_id':              route_id,
                    'schedule_relationship': sched_rel,
                    'stop_sequence':         stu.stop_sequence,
                    'stop_id':               stu.stop_id,
                    'predicted_unix':        predicted_time,
                    'predicted_ts':          datetime.fromtimestamp(
                                                 predicted_time, tz=timezone.utc
                                             ).isoformat(),
                })

    # Append this day's records to DuckDB and free the Python list
    if records:
        df = pd.DataFrame(records)
        con.execute('INSERT INTO gtfsrt SELECT * FROM df')
        total_records_inserted += len(records)
        print(f'  {service_date_str}: {len(day_files):4d} files, {len(records):>9,} records')
        del df, records
    else:
        print(f'  {service_date_str}: {len(day_files):4d} files, 0 records (all failed?)')

print()
print(f'=== Parse complete ===')
print(f'Total records inserted: {total_records_inserted:,}')
print(f'Failed files:           {len(failed_files)}')
if failed_files:
    print('First 10 failures:')
    for name, err in failed_files[:10]:
        print(f'  {name}: {err}')

Dates to process: 13
Dates skipped (already in table): 9



FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026-04-13: 1438 files, 21,631,354 records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026-04-14: 1438 files, 21,889,000 records


...

  2026-04-23: 1438 files, 22,487,055 records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026-04-24: 1439 files, 22,323,890 records


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

  2026-04-25:  678 files, 7,870,344 records

=== Parse complete ===
Total records inserted: 253,346,424
Failed files:           0


In [6]:
# These tables are recreated each run --- cheap operation, ensures consistency
# with the static GTFS files currently on disk

con.execute('DROP TABLE IF EXISTS trips')
con.execute(f"""
    CREATE TABLE trips AS
    SELECT * FROM read_csv_auto('{STATIC_DIR / 'trips.txt'}')
""")

con.execute('DROP TABLE IF EXISTS calendar')
con.execute(f"""
    CREATE TABLE calendar AS
    SELECT * FROM read_csv_auto('{STATIC_DIR / 'calendar.txt'}')
""")

con.execute('DROP TABLE IF EXISTS calendar_dates')
con.execute(f"""
    CREATE TABLE calendar_dates AS
    SELECT * FROM read_csv_auto('{STATIC_DIR / 'calendar_dates.txt'}')
""")

con.execute('DROP TABLE IF EXISTS stop_times')
con.execute(f"""
    CREATE TABLE stop_times AS
    SELECT
        trip_id,
        stop_sequence,
        stop_id,
        arrival_time,
        departure_time,
        timepoint,
        CAST(SPLIT_PART(arrival_time, ':', 1) AS INTEGER) * 3600 +
        CAST(SPLIT_PART(arrival_time, ':', 2) AS INTEGER) * 60 +
        CAST(SPLIT_PART(arrival_time, ':', 3) AS INTEGER) AS arrival_seconds,
        CAST(SPLIT_PART(departure_time, ':', 1) AS INTEGER) * 3600 +
        CAST(SPLIT_PART(departure_time, ':', 2) AS INTEGER) * 60 +
        CAST(SPLIT_PART(departure_time, ':', 3) AS INTEGER) AS departure_seconds
    FROM read_csv_auto('{STATIC_DIR / 'stop_times.txt'}')
""")

print('Static GTFS tables loaded:')
for tbl in ['trips', 'calendar', 'calendar_dates', 'stop_times']:
    n = con.sql(f'SELECT COUNT(*) AS n FROM {tbl}').fetchone()[0]
    print(f'  {tbl:18s} {n:>10,} rows')

Static GTFS tables loaded:
  trips                  35,316 rows
  calendar                   28 rows
  calendar_dates            688 rows
  stop_times          2,045,404 rows


In [7]:
print('=== gtfsrt table summary ===')
print()

print('Row count and date range:')
print(con.sql("""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT service_date) AS distinct_dates,
        MIN(service_date) AS first_date,
        MAX(service_date) AS last_date
    FROM gtfsrt
""").df().to_string(index=False))
print()

print('Records per service_date:')
print(con.sql("""
    SELECT service_date, COUNT(*) AS records
    FROM gtfsrt
    GROUP BY service_date
    ORDER BY service_date
""").df().to_string(index=False))
print()

print('Schedule_relationship distribution (0 = SCHEDULED):')
print(con.sql("""
    SELECT
        schedule_relationship,
        COUNT(*) AS n,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2) AS pct
    FROM gtfsrt
    GROUP BY schedule_relationship
    ORDER BY schedule_relationship
""").df().to_string(index=False))
print()

print('Top 10 routes by record count (focus routes 23 and 47):')
print(con.sql("""
    SELECT route_id, COUNT(*) AS records
    FROM gtfsrt
    GROUP BY route_id
    ORDER BY records DESC
    LIMIT 10
""").df().to_string(index=False))

=== gtfsrt table summary ===

Row count and date range:
 total_rows  distinct_dates first_date  last_date
  402682884              22 2026-04-04 2026-04-25

Records per service_date:
service_date  records
  2026-04-04  3149556
  2026-04-05 10997311
  2026-04-06 21412314
  2026-04-07 21872644
  2026-04-08 21738662
  2026-04-09 21822929
  2026-04-10 22220330
  2026-04-11 14810364
  2026-04-12 11312350
  2026-04-13 21631354
  2026-04-14 21889000
  2026-04-15 22160498
  2026-04-16 22048695
  2026-04-17 22058712
  2026-04-18 14746869
  2026-04-19 10999845
  2026-04-20 21638070
  2026-04-21 21639051
  2026-04-22 21853041
  2026-04-23 22487055
  2026-04-24 22323890
  2026-04-25  7870344

Schedule_relationship distribution (0 = SCHEDULED):
 schedule_relationship         n   pct
                     0 402682884 100.0

Top 10 routes by record count (focus routes 23 and 47):
route_id  records
      23 12558025
      47 12451384
      63 10943394
      57 10798530
      18 10555545
     113  89133